# 1. Data Preparation for Model Training

This notebook handles the final data preparation steps. It takes pre-filtered and imputed omics data files as input and creates a unified, model-ready dataset for training.

*Note: Initial quality control (QC), filtering, and imputation for the raw genotype, proteome, and metabolite data were performed in a separate, prior process.*

**Key Steps:**
1.  **Load Pre-processed Data:** Read the pre-filtered data files for genotype, proteome, metabolite, and clinical features.
2.  **Filter Common Samples:** Identify and retain only the samples present across all datasets.
3.  **Sex-specific Normalization:** Split the data by sex, apply Z-score normalization to each group, and then merge them back together.
4.  **Tensor Conversion:** Convert the cleaned and normalized Pandas DataFrames into PyTorch Tensors.
5.  **Save Final Dataset:** Store the final tensors and the clinical DataFrame in a single `.pt` file for easy access in the next notebook.

### 1.1. Import Libraries and Configuration


In [34]:
import sys
import pandas as pd
import torch

# Add the project's 'src' directory to the Python path
sys.path.append('/data02/jaejoon/T2D_subtype_analysis/src')

import config
from utils import normalize, set_seed

# Set the random seed for reproducibility
set_seed(config.RANDOM_STATE)

print("Libraries and configuration loaded successfully.")

Libraries and configuration loaded successfully.


### 1.2. Load Raw Data


In [ ]:
print("Loading data...")

# Load each data file using the paths defined in config.py
genotype_df = pd.read_csv(config.GENOTYPE_PATH)
proteome_df = pd.read_csv(config.PROTEOME_PATH)
metabolite_df = pd.read_csv(config.METABOLITE_PATH)
clinical_df = pd.read_csv(config.CLINICAL_PATH)

# Standardize ID columns to string type for consistent merging
genotype_df["FID"] = genotype_df["FID"].astype(str)
genotype_df["IID"] = genotype_df["IID"].astype(str)
proteome_df["id"] = proteome_df["id"].astype(str)
metabolite_df["id"] = metabolite_df["id"].astype(str)
clinical_df["id"] = clinical_df["id"].astype(str)

print("Data loading complete.")
print(f"Genotype data shape: {genotype_df.shape}")
print(f"Proteome data shape: {proteome_df.shape}")
print(f"Metabolite data shape: {metabolite_df.shape}")
print(f"Clinical data shape: {clinical_df.shape}")


Loading data...
Data loading complete.
Genotype data shape: (857, 424)
Proteome data shape: (858, 715)
Metabolite data shape: (890, 295)
Clinical data shape: (716, 8)


### 1.3. Filter by Common Samples


In [36]:
# Find the set of common sample IDs present in all four datasets
common_ids = set(genotype_df["IID"].values) & set(proteome_df["id"].values) & set(metabolite_df["id"].values) & set(clinical_df["id"].values)

print(f"Found {len(common_ids)} common samples.")

# Filter and sort each dataframe to include only common samples
genotype_df = genotype_df[genotype_df["IID"].isin(common_ids)].sort_values(by="IID").reset_index(drop=True)
proteome_df = proteome_df[proteome_df["id"].isin(common_ids)].sort_values(by="id").reset_index(drop=True)
metabolite_df = metabolite_df[metabolite_df["id"].isin(common_ids)].sort_values(by="id").reset_index(drop=True)
clinical_df = clinical_df[clinical_df["id"].isin(common_ids)].sort_values(by="id").reset_index(drop=True)

print("All dataframes have been filtered and sorted.")


Found 670 common samples.
All dataframes have been filtered and sorted.


### 1.4. Sex-specific Splitting and Normalization


In [37]:
# As per the paper's methodology, data is normalized separately for males and females.
# In the sample data, 1 represents Male and 2 represents Female.
male_idx = clinical_df["sex"] == 1
female_idx = clinical_df["sex"] == 2

# --- Genotype Data ---
female_geno = genotype_df[genotype_df["IID"].isin(clinical_df[female_idx]["id"])].iloc[:, [1] + list(range(6, genotype_df.shape[1]))] # PLINK format: first 6 columns are metadata
male_geno = genotype_df[genotype_df["IID"].isin(clinical_df[male_idx]["id"])].iloc[:, [1] + list(range(6, genotype_df.shape[1]))]
# Remove SNPs with zero variance within each sex
female_geno = pd.concat([female_geno.iloc[:, 0], female_geno.iloc[:, 1:].loc[:, female_geno.iloc[:, 1:].std() != 0]], axis=1)
male_geno = pd.concat([male_geno.iloc[:, 0], male_geno.iloc[:, 1:].loc[:, male_geno.iloc[:, 1:].std() != 0]], axis=1)
# Filter by common SNPs between sexes
common_snps = female_geno.columns.intersection(male_geno.columns)
female_geno = female_geno.loc[:, common_snps]
male_geno = male_geno.loc[:, common_snps]
# Normalize
female_geno_norm = pd.concat([female_geno.iloc[:,0], normalize(female_geno.iloc[:,1:])], axis=1)
male_geno_norm = pd.concat([male_geno.iloc[:,0], normalize(male_geno.iloc[:,1:])], axis=1)

# --- Proteome Data ---
female_prot = proteome_df[female_idx]
male_prot = proteome_df[male_idx]
female_prot_norm = pd.concat([female_prot.iloc[:,0], normalize(female_prot.iloc[:,1:])], axis=1)
male_prot_norm = pd.concat([male_prot.iloc[:,0], normalize(male_prot.iloc[:,1:])], axis=1)

# --- Metabolite Data ---
female_metab = metabolite_df[female_idx]
male_metab = metabolite_df[male_idx]
female_metab_norm = pd.concat([female_metab.iloc[:,0], normalize(female_metab.iloc[:,1:])], axis=1)
male_metab_norm = pd.concat([male_metab.iloc[:,0], normalize(male_metab.iloc[:,1:])], axis=1)

# --- Clinical Data ---
female_clin = clinical_df[female_idx]
male_clin = clinical_df[male_idx]
# Normalize only the numericbclinical variables used for model output
female_clin_norm = pd.concat([female_clin.iloc[:,0], normalize(female_clin[["bmi",	"hba1c",	"age_at_diagnosis",	"HOMA_B",	"HOMA_IR"]])], axis=1)
male_clin_norm = pd.concat([male_clin.iloc[:,0], normalize(male_clin[["bmi",	"hba1c",	"age_at_diagnosis",	"HOMA_B",	"HOMA_IR"]])], axis=1)

print("Sex-specific splitting and normalization complete.")


Sex-specific splitting and normalization complete.


In [38]:
# Recombine male and female data and sort by ID
genotype_final = pd.concat([female_geno_norm, male_geno_norm]).sort_values("IID").reset_index(drop=True)
proteome_final = pd.concat([female_prot_norm, male_prot_norm]).sort_values("id").reset_index(drop=True)
metabolite_final = pd.concat([female_metab_norm, male_metab_norm]).sort_values("id").reset_index(drop=True)
clinical_final = pd.concat([female_clin_norm, male_clin_norm]).sort_values("id").reset_index(drop=True)

# Also create a final, sorted version of the original (unnormalized) clinical dataframe
clinical_df_final = pd.concat([female_clin, male_clin]).sort_values(by="id").reset_index(drop=True)

# Convert Pandas DataFrames to PyTorch Tensors for model input
# Exclude ID columns, keeping only the numerical data
input_genotype = torch.tensor(genotype_final.iloc[:, 1:].values, dtype=torch.float32)
input_proteome = torch.tensor(proteome_final.iloc[:, 1:].values, dtype=torch.float32)
input_metabolite = torch.tensor(metabolite_final.iloc[:, 1:].values, dtype=torch.float32)
output_clinical = torch.tensor(clinical_final.iloc[:, 1:].values, dtype=torch.float32)

print("Tensor conversion complete.")
print(f"Final genotype tensor shape: {input_genotype.shape}")
print(f"Final proteome tensor shape: {input_proteome.shape}")
print(f"Final metabolite tensor shape: {input_metabolite.shape}")
print(f"Final clinical tensor shape: {output_clinical.shape}")


Tensor conversion complete.
Final genotype tensor shape: torch.Size([670, 415])
Final proteome tensor shape: torch.Size([670, 714])
Final metabolite tensor shape: torch.Size([670, 294])
Final clinical tensor shape: torch.Size([670, 5])


### 1.5. Generate Benchmark Labels using K-means

We perform K-means clustering on the final clinical variables to create benchmark labels, following the methodology proposed by Ahlqvist et al. (2018).  
These labels will serve as a ground truth reference for evaluating our multi-omics model in later notebooks.

In [39]:
# We perform K-means clustering on the final clinical variables to create benchmark labels.
print("Generating benchmark labels...")

from sklearn.cluster import KMeans

# Initialize and fit the K-means model on the normalized clinical data
kmeans = KMeans(n_clusters=config.NUM_CLUSTERS, init="k-means++", n_init=100, random_state=config.RANDOM_STATE)
benchmark_labels = kmeans.fit_predict(output_clinical.numpy())

# Add numeric cluster IDs to the final (unnormalized) clinical dataframe
clinical_df_final['kmeans_cluster_id'] = benchmark_labels

# Map the numeric cluster IDs to subtype names (SIDD, SIRD, etc.)
# This mapping uses the unnormalized clinical values for interpretability.
features = ["HOMA_IR", "hba1c", "bmi", "age_at_diagnosis"]
subtypes = ["SIRD", "SIDD", "MOD", "MARD"] # Order must match features!

mapping = {}
# Note: groupby column is 'kmeans_cluster_id' here
mean_values = clinical_df_final.groupby('kmeans_cluster_id')[features].mean()
assigned_clusters = set()

print("Mapping clusters to subtypes...")
for feature, subtype in zip(features, subtypes):
    # Find the cluster with the maximum value for this feature
    max_cluster = mean_values[feature].idxmax()
    
    # Check for conflict
    if max_cluster in assigned_clusters:
        print(f"Warning: Cluster {max_cluster} (best for {subtype}) has already been assigned.")
        mapping = None
        break
        
    mapping[int(max_cluster)] = subtype
    assigned_clusters.add(int(max_cluster))
    print(f"  - {subtype} assigned to Cluster {max_cluster} (Max {feature})")

# Apply mapping if successful
if mapping is not None and len(mapping) == config.NUM_CLUSTERS:
    clinical_df_final["kmeans_cluster"] = clinical_df_final["kmeans_cluster_id"].map(mapping)
    print("\nBenchmark clustering and subtype mapping complete.")
    print("Subtype counts:\n", clinical_df_final["kmeans_cluster"].value_counts())
else:
    print("\nError: Failed to map all clusters uniquely. Please check 'mean_values'.")

Generating benchmark labels...
Mapping clusters to subtypes...
  - SIRD assigned to Cluster 3 (Max HOMA_IR)
  - SIDD assigned to Cluster 1 (Max hba1c)
  - MOD assigned to Cluster 0 (Max bmi)
  - MARD assigned to Cluster 2 (Max age_at_diagnosis)

Benchmark clustering and subtype mapping complete.
Subtype counts:
 kmeans_cluster
MARD    269
MOD     178
SIDD    169
SIRD     54
Name: count, dtype: int64


### 1.6. Save Final Dataset


In [40]:
# Package all processed data, now including the benchmarked clinical_df, into a dictionary
processed_data = {
    'input_genotype': input_genotype,
    'input_proteome': input_proteome,
    'input_metabolite': input_metabolite,
    'output_clinical': output_clinical,
    'clinical_df': clinical_df_final, # This now contains the benchmark labels
    'genotype_features': genotype_final.columns[1:].tolist(),
    'proteome_features': proteome_final.columns[1:].tolist(),
    'metabolite_features': metabolite_final.columns[1:].tolist(),
}

# Create the directory for processed data if it doesn't exist
config.PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Save the dictionary to a single file
save_path = config.PROCESSED_DATA_DIR / "processed_dataset.pt"
torch.save(processed_data, save_path)

print(f"\nProcessed data saved to: \n{save_path}")
print("Feature names and benchmark labels are now included in the saved file.")



Processed data saved to: 
/data02/jaejoon/T2D_subtype_analysis/data/processed/processed_dataset.pt
Feature names and benchmark labels are now included in the saved file.
